# CUPED covariate check, pooled, no arm comparisons

CUPED's covariate is locked as history. The open question is whether adding the other pre-period variables (a multi-covariate regression adjustment) would cut variance enough to be worth reporting beside it.

CUPED's gain is how much of the outcome the covariates predict. With one covariate the variance shrinks by a factor of (1 - rho^2). With several it shrinks by (1 - R^2), where R^2 comes from regressing the outcome on all of them. So this notebook measures R^2 for history alone and for all pre-period variables together, for conversion and for spend.

Everything is pooled across all 64,000 customers. The arm column is dropped right after loading, so no outcome is compared across arms here.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from load import load_hillstrom

df = load_hillstrom()
df = df.drop(columns=["segment"])
assert "segment" not in df.columns

outcomes = ["conversion", "spend"]
smallest_arm = 21306
print(f"rows: {len(df)}, columns: {list(df.columns)}")

rows: 64000, columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'visit', 'conversion', 'spend']


## Step 1. Pre-period design matrix

Pre-period variables, all measured before the send.

- numeric, recency, history
- 0/1 already, mens, womens, newbie
- zip_code and channel turned into 0/1 columns, dropping one level of each as the baseline so the columns are not redundant with the intercept

history_segment is left out because it is a binned copy of history, same as in the balance check.

In [2]:
X_all = pd.get_dummies(
    df[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]],
    columns=["zip_code", "channel"], drop_first=True, dtype=float,
)
print(f"covariate columns ({X_all.shape[1]}): {list(X_all.columns)}")

covariate columns (9): ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


## Step 2. R^2 from an ordinary least squares fit

For an outcome y and covariates X (with an intercept column added), least squares finds the coefficients b that minimize the sum of squared residuals. Then

- residuals e = y - X b
- R^2 = 1 - sum(e^2) / sum((y - mean(y))^2)
- adjusted R^2 = 1 - (1 - R^2) (n - 1) / (n - k - 1), with k covariates, which penalizes extra columns

R^2 is the share of the outcome's variance the covariates explain, which is the share CUPED would remove.

In [3]:
def ols_r2(y, X):
    A = np.column_stack([np.ones(len(X)), X.to_numpy(dtype=float)])
    b, *_ = np.linalg.lstsq(A, y, rcond=None)
    e = y - A @ b
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    n, k = len(y), X.shape[1]
    adj = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    return r2, adj

## Step 3. history alone, R^2 vs rho^2

With one covariate, R^2 from the regression equals the squared correlation rho^2. Computing both is a check that the regression code is right. The Session 0 pooled correlations were 0.029405 (conversion) and 0.021729 (spend).

In [4]:
rows = []
for y_name in outcomes:
    y = df[y_name].to_numpy(dtype=float)
    rho = np.corrcoef(y, df["history"])[0, 1]
    r2, adj = ols_r2(y, df[["history"]])
    print(f"{y_name}: rho {rho:.6f}, rho^2 {rho**2:.8f}, regression R^2 {r2:.8f}, adjusted {adj:.8f}")
    rows.append({"outcome": y_name, "covariates": "history only", "k": 1, "r2": r2, "adj_r2": adj})

conversion: rho 0.029405, rho^2 0.00086464, regression R^2 0.00086464, adjusted 0.00084903
spend: rho 0.021729, rho^2 0.00047215, regression R^2 0.00047215, adjusted 0.00045653


## Step 4. All pre-period variables together

Same fit with every column from Step 1.

In [5]:
for y_name in outcomes:
    y = df[y_name].to_numpy(dtype=float)
    r2, adj = ols_r2(y, X_all)
    print(f"{y_name}: R^2 {r2:.8f}, adjusted {adj:.8f}")
    rows.append({"outcome": y_name, "covariates": "all pre-period", "k": X_all.shape[1], "r2": r2, "adj_r2": adj})

conversion: R^2 0.00191993, adjusted 0.00177955
spend: R^2 0.00092935, adjusted 0.00078883


## Step 5. What the variance reduction is worth in customers

If CUPED removes a share R^2 of the variance, the standard error shrinks as if the sample were 1 / (1 - R^2) times larger. Applied to the smallest arm (No E-Mail, 21,306 customers from `00_power_and_design.ipynb`)

- variance reduction = R^2 (using adjusted R^2, the more cautious one)
- equivalent n = 21,306 / (1 - R^2)
- customers saved per arm = equivalent n - 21,306

In [6]:
res = pd.DataFrame(rows)
res["variance_reduction_pct"] = 100 * res["adj_r2"]
res["equivalent_n_smallest_arm"] = smallest_arm / (1 - res["adj_r2"])
res["customers_saved_per_arm"] = res["equivalent_n_smallest_arm"] - smallest_arm
res

,outcome,covariates,k,r2,adj_r2,variance_reduction_pct,equivalent_n_smallest_arm,customers_saved_per_arm
0,conversion,history only,1,0.000865,0.000849,0.084903,21324.104708,18.104708
1,spend,history only,1,0.000472,0.000457,0.045653,21315.731299,9.731299
2,conversion,all pre-period,9,0.001920,0.001780,0.177955,21343.982771,37.982771
3,spend,all pre-period,9,0.000929,0.000789,0.078883,21322.820108,16.820108


## Step 6. Gain from adding the other covariates

The extra variance reduction from going from history alone to all pre-period variables, per outcome.

In [7]:
piv = res.pivot(index="outcome", columns="covariates", values="adj_r2")
piv["extra_reduction_pct_points"] = 100 * (piv["all pre-period"] - piv["history only"])
piv

covariates,all pre-period,history only,extra_reduction_pct_points
outcome,,,
conversion,0.001780,0.000849,0.093053
spend,0.000789,0.000457,0.033230


## Save results

In [8]:
tables = Path.cwd().parent / "results" / "tables"
res.to_csv(tables / "cuped_covariate_check.csv", index=False)
print("saved results/tables/cuped_covariate_check.csv")

saved results/tables/cuped_covariate_check.csv